# CS604: ViT-B/16 trained and tested at 288 px (Colab)

Final-style training like `cs604_vit_final.ipynb`, set up for larger images. One "Run all" trains one full fine-tuning model and writes the val and test submission files for CodaBench.

> **Post-deadline and unranked.** This model was trained after Phase 1 closed, so it is not our Phase 1 model. The TA confirmed that we may submit its test predictions as long as the result does not enter the ranking. Our ranked Phase 2 result stays `1019_1319_1_full` (test 76.09%). Read "Next steps" at the end before uploading.

> **Current setting: predict only.** `LOAD_RUN = "1105_1032_1_full_288"` is the run this notebook trained (8 epochs at 288 px, 1.61 GPU-hours). `TEST_SIZE = 320` is the best test size on the holdout in `methods/09_full_288.ipynb` (run 1: 77.78% at 320 px vs 77.24% at 288 px). "Run all" downloads val and test, predicts them at 320 px with flip TTA, and writes the files to `cs604/runs/<BATCH_NAME>_predict_1105_1032_1_full_288_320px_tta/`. It takes about 10 minutes and trains nothing. Set `LOAD_RUN = None` to train a new run instead.

**Recipe**: the same as our final model `1019_1319_1_full` (CodaBench val 76.22%, test 76.09%), except for the image size and the training time.

| | `1019_1319_1_full` | This notebook (default) |
|---|---|---|
| Method and head | full fine-tuning, prototype head | same |
| Data | all 45 images per class | same |
| Schedule | 8 epochs, 1 warmup epoch, LR 1e-4 with layer decay 0.75, cosine to 0 | same |
| Training size | 224 px in all 8 epochs | **288 px in all 8 epochs** |
| Prediction | 224 px, flip TTA | **288 px** (`TEST_SIZE`), flip TTA |
| Training GPU-hours (A100-80GB) | 1.03 | **about 1.7** (estimate) |

**Why 288 px**: `methods/08_linear_exp.ipynb` tested a model trained at 224 px (`1019_1120_1_full`) at larger sizes, with no extra training. Its holdout top-1 went from 73.38% at 224 px to 75.50% at 288 px (+2.13 points). Training at 288 px lets the model also learn from the larger input. The model is created with `dynamic_img_size=True`, so timm interpolates the 14 × 14 position embeddings to the 18 × 18 patch grid of a 288 px image.

**Compute budget**: the brief's constraint is 2 hours × 48 GB VRAM (±10%): 1.2 hours on an A100-80GB, or 1.32 with the 10%. A 288 px epoch costs about 1.7× a 224 px epoch (325 tokens instead of 197), so the default plan needs about 1.7 GPU-hours, which is over the constraint. As a post-deadline experiment it ignores the constraint: `TIME_BUDGET_H = 3.0` only stops a run that is far too slow. State its GPU-hours in the report. Plans that respect the constraint:

| `TRAIN_SIZES` | `TIME_BUDGET_H` | GPU-hours (estimate) |
|---|---|---|
| `[288] * 8` (default) | `3.0` | about 1.7, over the constraint |
| `[224] * 6 + [288] * 2` | `1.32` | about 1.2, within the 10% |
| `[224] * 7 + [288]` | `None` (= 1.2) | about 1.1 |

After the first 120 steps, the notebook measures the speed and checks the plan against `TIME_BUDGET_H`. If the plan would not fit, it drops 224 px epochs first and keeps the 288 px ones.

**Choosing the test size**: after training at 288 px, a test size above 288 px may work better (the same effect that made 288 px the best size for the 224 px model). `methods/09_full_288.ipynb` measures this on the holdout set: use the best test size of its run 1 (288, 288, 288), or of its run 2 (224, 224, 288) for a mixed plan. Set `TEST_SIZE` before predicting, or predict again later without training (step 5 below). Choose the size only from holdout results, never from test scores.

**Time** for one "Run all" with the default plan (A100-80GB): about 15 minutes to download and extract train, val and test, about 1.7 hours of training (`1019_1319_1_full` at 224 px: 6.2 min for the head and about 6.9 min per epoch), and about 5 minutes for the predictions.

**How to use**
1. `Runtime → Change runtime type`: **A100 GPU**. Then `Runtime → Run all`, and allow Google Drive access.
2. The results go to `cs604/runs/<BATCH_NAME>_1_full_288/` on Drive: `submission_test_*.zip` (the file to upload), `submission_val_*.zip`, `train_log.txt`, `curves.png`, `summary.json`, and the weights `final.pt`.
3. Check the log before uploading: "[test] 100,000 predictions at 288 px over ... classes", and "[test] agreement with 1102_1534_predict_1019_1319_1_full_tta". The agreement compares the new test predictions with those of our final model, without labels. Two models that are both about 76% accurate must agree on at least 52% of the images, so a much lower number means something is wrong.
4. If the session disconnects: set `BATCH_NAME` to the name printed at the start of the log, and "Run all" again. The run continues from its last finished epoch.
5. To predict again at another size without training: set `LOAD_RUN` to the run name and `TEST_SIZE` to the new size, then "Run all". The training set is not downloaded, and the files go to a new folder `cs604/runs/<BATCH_NAME>_predict_<run>_<size>px_tta/`.

**Not in this notebook**: the other methods, the lp and random heads, `torch.compile`, and T4/CPU support; they are in `cs604_vit_final.ipynb`. Its `PHASE2_TEST` switch predicts with the ranked model, so it is not used here.

## 1. Config

The defaults below are the recipe of `1019_1319_1_full` plus the image sizes. `TRAIN_SIZES` has one size per epoch, so its length must equal `EPOCHS`. All sizes must be multiples of 16, the patch size.

In [ ]:
TEAM_ID = "06"          # team number; the submission files are val_team_06.txt and test_team_06.txt

# ===== Current setting: predict only (LOAD_RUN below): the trained run 1105_1032_1_full_288, at TEST_SIZE = 320 px =====
# ===== With LOAD_RUN = None: train this run, the recipe of our final model 1019_1319_1_full at 288 px =====
EXPERIMENTS = [
    dict(TRAIN_SIZES=[288] * 8),
]
BATCH_NAME = None       # None = auto name; to resume after a disconnect, set the BATCH_NAME printed at the start of the log
LOAD_RUN = "1105_1032_1_full_288"  # trained run: skip training, load its final.pt and only predict; None = train EXPERIMENTS
PREDICT_SPLITS = ["val", "test"]  # val as a check, test for the CodaBench submission
COMPARE_RUN = "1102_1534_predict_1019_1319_1_full_tta"  # folder in cs604/runs/ with the final model's files, for the agreement check

# ===== Image sizes in pixels, multiples of 16 (the patch size) =====
TRAIN_SIZES = [288] * 8  # training size of each epoch: [288] * 8 about 1.7 GPU-hours; [224] * 6 + [288] * 2 about 1.2;
                         # [224] * 7 + [288] about 1.1 (estimates for an A100-80GB)
TEST_SIZE = 320         # size of the val/test predictions: the best size on the holdout in methods/09_full_288.ipynb run 1
                        # (77.78% at 320 px vs 77.24% at 288 px); 1105_1032_1_full_288 was first predicted at 288 px
TTA_FLIP = True         # also predict the mirrored image and add the logits, as for the final model (doubles the inference cost)

# ===== Training recipe: the same as 1019_1319_1_full =====
METHOD = "full"         # full fine-tuning only in this notebook
MODEL_NAME = "vit_base_patch16_224.augreg2_in21k_ft_in1k"  # timm version given on page 1 of the doc (IN21K→IN1K)
HOLDOUT_PER_CLASS = 0   # 0 = train on all 45 images per class, as the final model; 2 = keep the usual 20,000-image holdout
EPOCHS = 8              # must equal len(TRAIN_SIZES)
WARMUP_EPOCHS = 1       # the LR rises linearly for this many epochs, then follows a cosine down to 0
TIME_BUDGET_H = 3.0     # post-deadline run: ignores the course constraint and only stops a run that is far too slow;
                        # None = the constraint 96 GB·h ÷ VRAM (1.2 h on an A100-80GB), 1.32 = the constraint + 10%
BATCH_SIZE = None       # images per step; None = pick based on VRAM (256 on a 40 or 80 GB GPU)
EFFECTIVE_BATCH = 256   # if one step can't fit it, gradients are accumulated over several steps to reach it
LR = 1e-4
WEIGHT_DECAY = 0.05
LAYER_DECAY = 0.75      # layer-wise LR decay: the last block gets LR × 0.75, the first block LR × 0.75^12
DROP_PATH = 0.1         # stochastic depth: residual branches are randomly skipped during training
LABEL_SMOOTHING = 0.1
RRC_SCALE = (0.25, 1.0)  # area range for RandomResizedCrop (training augmentation)
LOG_EVERY = 100         # write a log line every N optimizer steps
SEED = 0
NUM_CLASSES = 10000

# ===== Data and paths =====
HF_REPO = "doem1997/cs604-course-data"       # the course dataset on Hugging Face; each session downloads it from here
DRIVE_DIR = "/content/drive/MyDrive/cs604"   # results and checkpoints are saved here; COMPARE_RUN and LOAD_RUN are read from here
DATA_DIR = "/content/data"                   # each session puts the data on the local Colab disk; training reads images from here
DISCONNECT_WHEN_DONE = False  # release the runtime when done to stop using compute units (handy when running in the background)

## 2. Setup

Install the libraries, mount Google Drive, check the GPU (it must support bf16: A100, L4 or H100), and set up the log. All outputs are saved on Drive, so nothing is lost if the session disconnects.

In [ ]:
# Colab already has torch / torchvision; install timm here, and hf_xet for fast parallel downloads from Hugging Face
!pip install -q timm hf_xet

In [ ]:
# Settings an experiment in EXPERIMENTS may override
RUN_KEYS = ("MODEL_NAME", "HOLDOUT_PER_CLASS", "EPOCHS", "WARMUP_EPOCHS", "TIME_BUDGET_H", "BATCH_SIZE",
            "EFFECTIVE_BATCH", "LR", "WEIGHT_DECAY", "LAYER_DECAY", "DROP_PATH", "LABEL_SMOOTHING", "RRC_SCALE",
            "LOG_EVERY", "SEED", "NUM_CLASSES", "TRAIN_SIZES", "TEST_SIZE", "TTA_FLIP")
BASE_CONFIG = {k: globals()[k] for k in RUN_KEYS}

import os, re, gc, json, math, time, random, shutil, zipfile, subprocess
from datetime import datetime
from types import SimpleNamespace

import numpy as np
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import timm
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
from google.colab import drive

# ---- Mount Google Drive: data is read from here, and results are saved here, so nothing is lost on disconnect ----
drive.mount("/content/drive")
OUT_ROOT = DRIVE_DIR

# ---- GPU: needs bf16 support, i.e. an A100, L4 or H100 runtime ----
assert torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8, (
    "Pick a GPU runtime with bf16 support: A100, L4 or H100")
DEVICE = "cuda"
props = torch.cuda.get_device_properties(0)
GPU_NAME = props.name
gib = props.total_memory / 1024**3
VRAM_GB = next((v for v in (8, 12, 16, 24, 32, 40, 48, 80, 94, 96, 141, 192) if v >= 0.98 * gib), round(gib))  # nominal VRAM
torch.backends.cuda.matmul.allow_tf32 = True  # TF32 for float32 matrix multiplies: much faster, accuracy barely changes
torch.backends.cudnn.allow_tf32 = True
NUM_WORKERS = min(16, os.cpu_count())  # processes that read and decode images in parallel
VERSIONS = f"torch {torch.__version__} (CUDA {torch.version.cuda}), timm {timm.__version__}"

os.environ["TZ"] = "Asia/Singapore"  # the Colab VM runs on UTC; use local time for batch names and log timestamps
time.tzset()
BATCH_NAME = BATCH_NAME or datetime.now().strftime("%m%d_%H%M")  # runs are named <BATCH_NAME>_<i>_full_<sizes>
BATCH_DIR = os.path.join(OUT_ROOT, "batches", BATCH_NAME)
os.makedirs(BATCH_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

LOG_FILE = None  # train_log.txt of the current run, set by run_experiment


def log(*args):
    """Print, and also append to this batch's batch_log.txt and the current run's train_log.txt (the report needs the training log)"""
    msg = " ".join(str(a) for a in args)
    print(msg, flush=True)
    stamp = datetime.now().strftime("[%Y-%m-%d %H:%M:%S] ")
    for path in (os.path.join(BATCH_DIR, "batch_log.txt"), LOG_FILE):
        if path:
            with open(path, "a") as f:
                f.write(stamp + msg + "\n")


ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
log(f"Batch {BATCH_NAME} -> {BATCH_DIR}  (to resume after a disconnect: set BATCH_NAME = \"{BATCH_NAME}\")")
log(f"Device {GPU_NAME} ({VRAM_GB} GB), bfloat16, {os.cpu_count()} CPUs, {ram_gb:.0f} GB RAM, "
    f"{shutil.disk_usage(DATA_DIR).free / 1e9:.0f} GB free disk")
log(f"Versions: {VERSIONS}")

## 3. Prepare the data

- Each new session downloads the data from Hugging Face to the local Colab disk (`DATA_DIR`): `train` for training, plus each split in `PREDICT_SPLITS` (val and test). The images are then read from the local disk.
- On Colab the download runs at about 500 MB/s. Each archive is deleted after it is extracted; while `train.tar.gz` is extracted, the disk needs about 80 GB free.
- With `LOAD_RUN` set, the training set is not needed, so it is not downloaded.
- When a split is ready, a marker file is written, so this step is skipped if you run it again in the same session.

In [ ]:
def prepare_split(split):
    """Download {split}.txt and {split}.tar.gz from Hugging Face and extract the images to the local Colab disk under DATA_DIR.
    Returns the path of the local {split}.txt"""
    list_path = os.path.join(DATA_DIR, f"{split}.txt")
    marker = os.path.join(DATA_DIR, f".{split}_ready")
    if os.path.exists(marker):  # already done in this session
        return list_path
    t0 = time.time()
    # val.txt and test.txt are plain (non-Xet) files on the Hub; ask for them uncompressed, because some
    # huggingface_hub/httpx versions fail to decode compressed responses
    hf_hub_download(HF_REPO, f"{split}.txt", repo_type="dataset", local_dir=DATA_DIR, headers={"Accept-Encoding": "identity"})
    tar_path = hf_hub_download(HF_REPO, f"{split}.tar.gz", repo_type="dataset", local_dir=os.path.join(DATA_DIR, "hf"))
    log(f"[{split}] downloaded from Hugging Face in {(time.time() - t0) / 60:.1f} min")
    log(f"[{split}] extracting {tar_path} ...")
    subprocess.run(["tar", "-xzf", tar_path, "-C", DATA_DIR], check=True)
    os.remove(tar_path)  # the images are out; free the disk space the archive takes (40 GB for train)
    with open(list_path) as f:
        n_expected = sum(1 for line in f if line.strip())
    n_found = sum(len(files) for _, _, files in os.walk(os.path.join(DATA_DIR, split)))
    assert n_found == n_expected, f"{split}: found {n_found} images, expected {n_expected}"
    open(marker, "w").close()
    log(f"[{split}] {n_found} images ready in {(time.time() - t0) / 60:.1f} min")
    return list_path


def read_list(path):
    """Each line is '<image path> <class id>', e.g. 'train/class_0000/9aef28bf-....jpg 0'; val and test have no class (-1)"""
    items = []
    with open(path) as f:
        for line in f:
            parts = line.split()
            if parts:
                items.append((parts[0], int(parts[1]) if len(parts) > 1 else -1))
    return items


TRAIN_ALL = [] if LOAD_RUN else read_list(prepare_split("train"))  # the training set is not needed when only predicting
PRED_ITEMS = {split: read_list(prepare_split(split)) for split in PREDICT_SPLITS}
log(f"train {len(TRAIN_ALL):,} images | " + " | ".join(f"{s} {len(v):,}" for s, v in PRED_ITEMS.items())
    + f" | {NUM_WORKERS} loader workers")

## 4. Model: full fine-tuning with dynamic image size

- `create_model` loads the pretrained ViT-B/16 with `dynamic_img_size=True`, and replaces the 1,000-class head with a new 10,000-class head.
- With `dynamic_img_size=True`, the stored 14 × 14 position embeddings are resized by bicubic interpolation to the patch grid of each input, in every forward pass (18 × 18 at 288 px). They stay 14 × 14 in the weights and are trained through the interpolation. At 224 px they are used as they are, so a 224 px epoch trains exactly the same model as `cs604_vit_final.ipynb`.
- All 85,798,656 backbone parameters and the 7,690,000 head parameters are trained. The image size does not change the number of parameters.
- `resolve_config` checks the sizes. With `TIME_BUDGET_H = None` it sets the course constraint as the time budget: 96 GB·h ÷ the GPU memory, so 1.2 hours on an A100-80GB.

In [ ]:
def is_head(name):
    return name in ("head.weight", "head.bias")


def resolve_config(overrides):
    """Defaults from section 1 + one experiment's overrides → the full config of this run"""
    unknown = set(overrides) - set(RUN_KEYS)
    assert not unknown, f"unknown keys in experiment config: {sorted(unknown)}"
    cfg = SimpleNamespace(**{**BASE_CONFIG, **overrides})
    assert len(cfg.TRAIN_SIZES) == cfg.EPOCHS, f"TRAIN_SIZES needs one size per epoch: {cfg.TRAIN_SIZES} for {cfg.EPOCHS} epochs"
    sizes = [*cfg.TRAIN_SIZES, cfg.TEST_SIZE]
    assert all(s % 16 == 0 for s in sizes), f"image sizes must be multiples of the 16 px patch size: {sizes}"
    if cfg.BATCH_SIZE is None:
        cfg.BATCH_SIZE = 256 if VRAM_GB >= 40 else 128 if VRAM_GB >= 24 else 64
    cfg.ACCUM_STEPS = max(1, cfg.EFFECTIVE_BATCH // cfg.BATCH_SIZE)
    if cfg.TIME_BUDGET_H is None:
        cfg.TIME_BUDGET_H = 96 / VRAM_GB  # 2 h × 48 GB = 96 GB·h
    return cfg


def create_model(cfg):
    """Pretrained ViT-B/16 from timm; its 1000-class ImageNet head is replaced by a new, randomly initialized 10,000-class head.
    dynamic_img_size=True: in every forward pass, the 14 × 14 position embeddings are interpolated to the patch grid of the
    input (18 × 18 at 288 px), so the model takes any size that is a multiple of 16. At 224 px they are used as they are"""
    return timm.create_model(cfg.MODEL_NAME, pretrained=True, num_classes=cfg.NUM_CLASSES,
                             drop_path_rate=cfg.DROP_PATH, dynamic_img_size=True).to(DEVICE)


def apply_method(net, cfg):
    """Full fine-tuning: every parameter of the backbone and the new head is trained"""
    return net.requires_grad_(True)


def param_report(net, ctx):
    """Trainable parameter counts. The 7.69M head is counted separately, as in the doc's LoRA reference (0.34%)"""
    n_train = sum(p.numel() for n, p in net.named_parameters() if p.requires_grad and not is_head(n))
    return {"backbone_params": ctx.n_backbone, "head_params": ctx.n_head, "head_mode": "fine-tuned",
            "trainable_excl_head": n_train, "trainable_pct": 100 * n_train / ctx.n_backbone,
            "trainable_incl_head": n_train + ctx.n_head}

## 5. Dataset and image transforms

- With `HOLDOUT_PER_CLASS = 0` (the default), all 450,000 training images are used, as for the final model. With 2, the same 20,000-image holdout as in the other notebooks is kept, and its accuracy is measured at `TEST_SIZE` after every epoch.
- The transforms take the size as an argument (`make_transform`). Training: a random crop (area range `RRC_SCALE`) resized to size × size, and a random flip. Prediction: the short side is resized to size / 0.9, then the center size × size is cropped (at 288 px: resize to 320, crop 288).

In [ ]:
def split_holdout(items, k, seed):
    """Hold out k images of every class: shuffle each class's image list with a fixed seed and take the first k"""
    by_class = {}
    for path, label in items:
        by_class.setdefault(label, []).append(path)
    rng = random.Random(seed)
    train, holdout = [], []
    for label in sorted(by_class):
        paths = sorted(by_class[label])
        rng.shuffle(paths)
        holdout += [(p, label) for p in paths[:k]]
        train += [(p, label) for p in paths[k:]]
    return train, holdout


class ImageList(Dataset):
    def __init__(self, items, transform):
        self.items, self.transform = items, transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        path, label = self.items[i]
        with Image.open(os.path.join(DATA_DIR, path)) as im:
            img = im.convert("RGB")
        return self.transform(img), label


BICUBIC = T.InterpolationMode.BICUBIC


def make_ctx(model, cfg):
    """Data setup for this run: train/holdout split, normalization values, the crop ratio for evaluation, param counts"""
    data_cfg = timm.data.resolve_model_data_config(model)
    train_items, holdout_items = split_holdout(TRAIN_ALL, cfg.HOLDOUT_PER_CLASS, cfg.SEED)
    return SimpleNamespace(
        train_items=train_items, holdout_items=holdout_items, crop_pct=data_cfg["crop_pct"],
        mean=torch.tensor(data_cfg["mean"], device=DEVICE).view(1, 3, 1, 1) * 255,
        std=torch.tensor(data_cfg["std"], device=DEVICE).view(1, 3, 1, 1) * 255,
        n_backbone=sum(p.numel() for n, p in model.named_parameters() if not is_head(n)),
        n_head=sum(p.numel() for p in model.head.parameters()),
    )


def make_transform(size, train, ctx, cfg):
    """Training: random crop resized to size × size, random flip. Evaluation: resize the short side to size / crop_pct,
    then crop the center size × size. Only crop/flip on the CPU and output uint8; normalize on the GPU to save CPU work"""
    if train:
        return T.Compose([T.RandomResizedCrop(size, scale=tuple(cfg.RRC_SCALE), interpolation=BICUBIC),
                          T.RandomHorizontalFlip(), T.PILToTensor()])
    return T.Compose([T.Resize(int(size / ctx.crop_pct), interpolation=BICUBIC), T.CenterCrop(size), T.PILToTensor()])


class Loader(DataLoader):
    """Before starting new worker processes, run garbage collection in the main process. Otherwise an old, interrupted DataLoader is copied into the new workers,
    and when it is collected there it prints lots of harmless AssertionError: can only test a child process messages."""

    def __iter__(self):
        gc.collect()
        return super().__iter__()


def make_loader(items, train, size, ctx, cfg):
    """Images of size × size px. Training: random crops, shuffled, BATCH_SIZE images per step.
    Evaluation: center crops, in order, 2 × BATCH_SIZE (no gradients, so it fits)"""
    return Loader(ImageList(items, make_transform(size, train, ctx, cfg)),
                  batch_size=cfg.BATCH_SIZE if train else 2 * cfg.BATCH_SIZE, shuffle=train, drop_last=train,
                  num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=train)


def to_input(x, ctx):
    """uint8 images (B, 3, size, size) → normalized float32 on the GPU"""
    return (x.to(DEVICE, non_blocking=True).float() - ctx.mean) / ctx.std


def autocast():
    """bf16 mixed precision: matrix multiplies run in bfloat16, the weights stay float32"""
    return torch.autocast("cuda", dtype=torch.bfloat16)


def free_gpu():
    """Free the GPU memory left by the previous run and reset the peak memory stats"""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

## 6. Classifier head initialization

If the new 10,000-class head starts from random weights, the gradients early in training damage the pretrained features. So the frozen backbone first extracts features for all training images, at the size of the first epoch (`TRAIN_SIZES[0]`, 288 px by default). For each class, the average direction of its features becomes the weight of that class (the class prototype), multiplied by a temperature. For the final model at 224 px this took 6.2 minutes; at 288 px it takes about 7–8 minutes. It counts toward the GPU time.

In [ ]:
class Clock:
    """Total GPU work time of a run: head init + training + holdout evaluation during training (not data prep)"""

    def __init__(self):
        self.used, self._t0 = 0.0, None

    def __enter__(self):
        self._t0 = time.time()
        return self

    def __exit__(self, *exc):
        torch.cuda.synchronize()
        self.used += time.time() - self._t0
        self._t0 = None

    @property
    def elapsed(self):
        return self.used + (time.time() - self._t0 if self._t0 else 0.0)


@torch.inference_mode()
def extract_features(net, items, size, ctx, cfg):
    """Features of the frozen backbone at size × size px: the class-token output after the final LayerNorm, 768 numbers per image"""
    net.eval()
    feats, labels = [], []
    for x, y in tqdm(make_loader(items, False, size, ctx, cfg), desc="train features", leave=False):
        with autocast():
            f = net.forward_head(net.forward_features(to_input(x, ctx)), pre_logits=True)
        feats.append(f.half())
        labels.append(y.to(DEVICE))
    return torch.cat(feats), torch.cat(labels)


@torch.inference_mode()
def class_prototypes(feats, labels, num_classes):
    """Mean direction of each class: L2-normalize every feature, add them up per class, normalize the sums"""
    protos = torch.zeros(num_classes, feats.shape[1], device=feats.device)
    for i in range(0, len(feats), 65536):
        protos.index_add_(0, labels[i:i + 65536], F.normalize(feats[i:i + 65536].float(), dim=1))
    return F.normalize(protos, dim=1)


@torch.inference_mode()
def fit_logit_scale(protos, feats, labels, n=10000):
    """Pick a good temperature for cosine similarity, so the softmax of the prototype head is not too flat or too sharp"""
    idx = torch.randperm(len(feats), device=feats.device)[:n]
    cos = F.normalize(feats[idx].float(), dim=1) @ protos.T
    losses = {s: F.cross_entropy(cos * s, labels[idx]).item() for s in (5, 10, 15, 20, 30, 40, 60, 80)}
    return min(losses, key=losses.get)


def init_head(model, ctx, cfg):
    """Prototype head: each class's weight is the mean direction of its training features, times a temperature.
    The features are extracted at the size of the first epoch, TRAIN_SIZES[0].
    Returns the seconds spent, feature extraction included; they count toward this run's GPU time"""
    t0 = time.time()
    feats, labels = extract_features(model, ctx.train_items, cfg.TRAIN_SIZES[0], ctx, cfg)
    protos = class_prototypes(feats, labels, cfg.NUM_CLASSES)
    scale = fit_logit_scale(protos, feats, labels)
    with torch.no_grad():
        # The head works on unnormalized features: divide by the mean feature norm so logits ≈ scale × cosine similarity
        model.head.weight.copy_(protos * (scale / feats.float().norm(dim=1).mean()))
        model.head.bias.zero_()
    torch.cuda.synchronize()
    seconds = time.time() - t0
    log(f"Head initialized from the class prototypes of {len(feats):,} train images at {cfg.TRAIN_SIZES[0]} px "
        f"(logit scale {scale}) "
        f"in {seconds / 60:.1f} min, counted in this run's GPU time")
    return seconds

## 7. Training

- As in `cs604_vit_final.ipynb`: AdamW, linear warmup for `WARMUP_EPOCHS` epochs and then a cosine decay to 0, layer-wise learning-rate decay, bf16 mixed precision, gradient clipping at 1.0, label smoothing 0.1.
- Epoch e trains at `TRAIN_SIZES[e]`. When the size changes, the training data loader is rebuilt with crops of the new size. The learning-rate schedule does not depend on the size. In a mixed plan such as `[224] * 7 + [288]`, the 288 px epoch therefore runs at the end of the cosine, from about 5% of the peak LR down to 0.
- **Time budget**: 20 steps after each size starts, the speed is measured over 100 steps. Then the time for the rest of the plan is estimated: epochs at other sizes are scaled by their compute (`token_cost`; 288 px costs about 1.7× more than 224 px), and 6% is added, because whole epochs run about 6% slower than these 100 steps. If the plan does not fit in 95% of `TIME_BUDGET_H`, later 224 px epochs are dropped first; if that is not enough, the LR schedule ends earlier. Training also stops at 105% of `TIME_BUDGET_H`. The log also shows the course constraint (96 GB·h ÷ VRAM, ±10%) for comparison.
- **Checkpoints**: `last.pt` is saved after every epoch (weights, optimizer, plan and history), so a disconnected run can continue from it.

In [ ]:
def build_param_groups(net, cfg):
    """Layer-wise LR decay: block i (1–12) gets LR × LAYER_DECAY^(13 − i); the patch embedding, class token and position
    embeddings count as layer 0, the final norm and the head get the full LR. No weight decay on 1-D params (biases,
    LayerNorm weights), the class token and the position embeddings"""
    num_layers = len(net.blocks) + 1
    groups = {}
    for n, p in net.named_parameters():
        m = re.match(r"blocks\.(\d+)\.", n)
        if m:
            layer = int(m.group(1)) + 1
        elif n.startswith(("cls_token", "pos_embed", "patch_embed")):
            layer = 0
        else:
            layer = num_layers
        lr = cfg.LR * cfg.LAYER_DECAY ** (num_layers - layer)
        wd = 0.0 if p.ndim <= 1 or n in ("cls_token", "pos_embed") else cfg.WEIGHT_DECAY
        groups.setdefault((lr, wd), []).append(p)
    return [{"params": ps, "lr": lr, "weight_decay": wd, "base_lr": lr} for (lr, wd), ps in groups.items()]


@torch.inference_mode()
def evaluate(net, loader, ctx):
    """Top-1 and top-5 accuracy on a labeled set"""
    net.eval()
    c1 = c5 = n = 0
    for x, y in loader:
        with autocast():
            top5 = net(to_input(x, ctx)).topk(5, dim=1).indices
        y = y.to(DEVICE)
        c1 += (top5[:, 0] == y).sum().item()
        c5 += (top5 == y[:, None]).any(dim=1).sum().item()
        n += len(y)
    return c1 / n, c5 / n


def save_state(net, path, **extra):
    """Save the trained params (all of them, about 370 MB) and the extra info. It is written to a temporary file first,
    so a disconnect while saving never leaves a broken file"""
    state = {n: p.detach().cpu() for n, p in net.named_parameters() if p.requires_grad or is_head(n)}
    torch.save({"state": state, **extra}, path + ".tmp")
    os.replace(path + ".tmp", path)


def token_cost(size):
    """Relative compute of one image at size px for ViT-B/16: the linear layers grow with the number of tokens N and the
    attention with N² (N = (size / 16)² + 1: 197 at 224 px, 325 at 288 px, so 288 px costs about 1.7× more)"""
    n = (size // 16) ** 2 + 1
    return n * (1 + n / 4608)


def remaining_seconds(plan, epoch, step, spe, rate):
    """GPU seconds for the rest of the plan from the current step, each epoch at its own size.
    rate = seconds per optimizer step per unit of token_cost, measured during training"""
    total = 0.0
    for e in range(epoch, len(plan["sizes"])):
        steps = min(plan["total"], (e + 1) * spe) - max(step, e * spe)
        total += max(0, steps) * rate * token_cost(plan["sizes"][e])
    return total


def fit_plan(plan, epoch, step, spe, rate, elapsed, budget_s, per_epoch_s):
    """Shorten the plan if it would not fit in 95% of the time budget. First drop later epochs at the smallest size, so the
    large-size epochs at the end are kept; if that is not enough, end the LR schedule earlier (as in cs604_vit_final.ipynb)"""
    limit = 0.95 * budget_s

    def need():
        return elapsed + remaining_seconds(plan, epoch, step, spe, rate) + per_epoch_s * (len(plan["sizes"]) - epoch)

    while need() > limit and len(plan["sizes"]) > epoch + 1 and min(plan["sizes"][epoch + 1:]) < max(plan["sizes"]):
        size = min(plan["sizes"][epoch + 1:])
        plan["sizes"].pop(plan["sizes"].index(size, epoch + 1))
        plan["total"] = len(plan["sizes"]) * spe
        log(f"Time budget: dropped one {size} px epoch; the plan is now {plan['sizes']}")
    if need() > limit:
        avail, fit = limit - elapsed - per_epoch_s * (len(plan["sizes"]) - epoch), step
        for e in range(epoch, len(plan["sizes"])):  # walk through the remaining steps until the time runs out
            steps = min(plan["total"], (e + 1) * spe) - max(step, e * spe)
            cost = rate * token_cost(plan["sizes"][e])
            k = max(0, min(steps, int(avail / cost)))
            fit, avail = fit + k, avail - k * cost
            if k < steps:
                break
        plan["total"] = max(fit, step + 1)
        if step < plan["warmup"]:
            plan["warmup"] = min(plan["warmup"], max(1, plan["total"] // 10))
        log(f"Time budget: the LR schedule now ends at step {plan['total']} ({plan['total'] / spe:.2f} epochs)")
    log(f"Plan: {plan['sizes']} px, about {need() / 3600:.2f} of the {budget_s / 3600:.2f} GPU-hour budget")


def train(model, ctx, cfg, run_dir, clock, resume_ckpt):
    """AdamW with linear warmup, then cosine decay to 0; bf16 autocast; gradient clipping at 1.0. Epoch e trains at
    plan["sizes"][e] px, and the loader is rebuilt when the size changes. Saves last.pt after every epoch and resumes from
    it. Returns (history, optimizer steps, steps per epoch, {training size: speed in img/s})"""
    spe = len(ctx.train_items) // (cfg.BATCH_SIZE * cfg.ACCUM_STEPS)  # optimizer steps per epoch
    plan = {"sizes": list(cfg.TRAIN_SIZES), "total": cfg.EPOCHS * spe, "warmup": max(1, int(cfg.WARMUP_EPOCHS * spe))}
    hist = {"step": [], "loss": [], "lr": [], "epoch": [], "train_size": [], "epoch_loss": [], "holdout_top1": [],
            "steps_per_epoch": spe}
    step, epoch, speeds = 0, 0, {}

    def lr_factor(s):
        if s < plan["warmup"]:
            return (s + 1) / plan["warmup"]
        return 0.5 * (1 + math.cos(math.pi * min(1.0, (s - plan["warmup"]) / max(1, plan["total"] - plan["warmup"]))))

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(build_param_groups(model, cfg), fused=True)
    if resume_ckpt:
        ck = torch.load(resume_ckpt, map_location="cpu", weights_only=False)
        _, unexpected = model.load_state_dict(ck["state"], strict=False)
        assert not unexpected, f"unexpected keys: {unexpected[:5]}"
        optimizer.load_state_dict(ck["optimizer"])
        step, epoch = ck["step"], ck["epoch"] + 1
        plan, hist, speeds, clock.used = ck["plan"], ck["hist"], ck["speeds"], ck["clock"]
        log(f"Resumed at epoch {epoch + 1}, step {step}, {clock.used / 3600:.2f} GPU-hours already used")
    holdout_loader = make_loader(ctx.holdout_items, False, cfg.TEST_SIZE, ctx, cfg) if ctx.holdout_items else None
    budget_s, per_epoch_s = cfg.TIME_BUDGET_H * 3600, (30 if holdout_loader else 5)
    log(f"Training {len(plan['sizes'])} epochs x {spe} steps (batch {cfg.BATCH_SIZE}x{cfg.ACCUM_STEPS}, warmup "
        f"{plan['warmup']} steps), sizes {plan['sizes']} px, time budget {cfg.TIME_BUDGET_H:.2f} h")
    train_loader = train_size = rate = None
    t_mark = 0.0
    while epoch < len(plan["sizes"]) and step < plan["total"]:
        if plan["sizes"][epoch] != train_size:
            train_size = plan["sizes"][epoch]
            train_loader = None  # stop the worker processes of the old loader before starting new ones
            train_loader = make_loader(ctx.train_items, True, train_size, ctx, cfg)
            speed_start = step + 20  # measure the speed again at the new size
            log(f"Epoch {epoch + 1}: training at {train_size} px ({(train_size // 16) ** 2 + 1} tokens per image)")
        model.train()
        optimizer.zero_grad(set_to_none=True)
        run_loss, run_n, ep_loss, ep_n, t_log = 0.0, 0, 0.0, 0, time.time()
        with clock:
            for i, (x, y) in enumerate(train_loader):
                with autocast():
                    loss = F.cross_entropy(model(to_input(x, ctx)), y.to(DEVICE, non_blocking=True),
                                           label_smoothing=cfg.LABEL_SMOOTHING)
                (loss / cfg.ACCUM_STEPS).backward()
                run_loss += loss.detach()
                run_n += 1
                if (i + 1) % cfg.ACCUM_STEPS:
                    continue  # gradient accumulation: add up the gradients of ACCUM_STEPS batches before each update
                for g in optimizer.param_groups:
                    g["lr"] = g["base_lr"] * lr_factor(step)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                step += 1

                # Speed at each size, over 100 steps that start 20 steps after the size is set; then check the time budget
                if step == speed_start:
                    t_mark = clock.elapsed
                elif step == speed_start + 100:
                    sec_per_step = (clock.elapsed - t_mark) / 100
                    speeds[train_size] = cfg.BATCH_SIZE * cfg.ACCUM_STEPS / sec_per_step
                    log(f"Speed at {train_size} px: {speeds[train_size]:.0f} img/s, {sec_per_step * spe / 60:.1f} min/epoch")
                    # whole epochs take about 6% longer than these 100 steps (1019_1319_1_full: 6.5 min/epoch here, 6.9 real)
                    rate = 1.06 * sec_per_step / token_cost(train_size)
                    fit_plan(plan, epoch, step, spe, rate, clock.elapsed, budget_s, per_epoch_s)

                if step % cfg.LOG_EVERY == 0:
                    avg = (run_loss / run_n).item()
                    if not math.isfinite(avg):
                        raise RuntimeError("Loss is NaN/inf; try a lower LR")
                    eta = f" | ETA {remaining_seconds(plan, epoch, step, spe, rate) / 60:.0f} min" if rate else ""
                    log(f"ep {epoch + 1}/{len(plan['sizes'])} ({train_size} px) step {step}/{plan['total']} | loss {avg:.4f} | "
                        f"lr {cfg.LR * lr_factor(step):.2e} | {run_n * cfg.BATCH_SIZE / (time.time() - t_log):.0f} img/s | "
                        f"{clock.elapsed / 60:.1f} min{eta}")
                    hist["step"].append(step)
                    hist["loss"].append(avg)
                    hist["lr"].append(cfg.LR * lr_factor(step))
                    ep_loss += avg * run_n
                    ep_n += run_n
                    run_loss, run_n, t_log = 0.0, 0, time.time()
                if step >= plan["total"] or clock.elapsed > 1.05 * budget_s:
                    break
            if run_n:
                ep_loss += float(run_loss)
                ep_n += run_n
            top1 = evaluate(model, holdout_loader, ctx)[0] if holdout_loader else None
        hist["epoch"].append(epoch + 1)
        hist["train_size"].append(train_size)
        hist["epoch_loss"].append(ep_loss / max(1, ep_n))
        hist["holdout_top1"].append(top1)
        log(f"== epoch {epoch + 1} (trained at {train_size} px): train loss {ep_loss / max(1, ep_n):.4f}"
            + (f", holdout top-1 at {cfg.TEST_SIZE} px {top1:.2%}" if top1 is not None else "")
            + f", {clock.elapsed / 60:.1f} min used")
        if clock.elapsed > 1.05 * budget_s and step < plan["total"]:
            plan["total"] = step  # out of time: stop here, also after a resume
            log(f"Time budget used up: training stops after {step} steps")
        save_state(model, os.path.join(run_dir, "last.pt"), optimizer=optimizer.state_dict(), step=step, epoch=epoch,
                   plan=plan, hist=hist, speeds=speeds, clock=clock.used)
        epoch += 1
    return hist, step, spe, speeds

## 8. Prediction, latency and submission files

- `predict` prepares every val/test image at `TEST_SIZE`. With `TTA_FLIP`, it adds the logits of the image and of its mirror image.
- `make_submission` writes `{split}_team_06.txt` (one line per image: path and class) and packs it into `submission_{split}_<run>.zip`. It checks that every image appears exactly once and that all classes are valid.
- `agreement` compares the new predictions with our final model's file in `COMPARE_RUN`: the share of images that get the same class. It needs no labels.
- Latency is measured at `TEST_SIZE` with random inputs: forward pass only, bf16, batch 1 run 100 times and batch 256 run 20 times. With flip TTA every image needs two forward passes, so the times are doubled.

In [ ]:
INK, MUTED, GRID, SURFACE, LINE = "#0b0b0b", "#52514e", "#e1e0d9", "#fcfcfb", "#2a78d6"


def style_axis(ax, title, xlabel, ylabel):
    ax.set_facecolor(SURFACE)
    ax.set_title(title, loc="left", color=INK, fontsize=11)
    ax.set_xlabel(xlabel, color=MUTED)
    ax.set_ylabel(ylabel, color=MUTED)
    ax.grid(True, color=GRID, linewidth=0.75)
    ax.set_axisbelow(True)
    ax.tick_params(colors=MUTED, length=0)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)


def plot_run_curves(hist, cfg, run_dir):
    """Save curves.png: the training loss against the epoch, with the epochs at a larger size shaded, and the holdout
    top-1 after each epoch if there is a holdout set"""
    if not hist["step"]:
        return
    accs = [(e, 100 * a) for e, a in zip(hist["epoch"], hist["holdout_top1"]) if a is not None]
    fig, axes = plt.subplots(1, 2 if accs else 1, figsize=(11 if accs else 6, 3.6), facecolor=SURFACE, squeeze=False)
    ax = axes[0][0]
    big = [e for e, s in enumerate(hist["train_size"]) if s != hist["train_size"][0]]
    for e in big:
        ax.axvspan(e, e + 1, color=GRID, linewidth=0)
    ax.plot([s / hist["steps_per_epoch"] for s in hist["step"]], hist["loss"], color=LINE, linewidth=1.5,
            solid_capstyle="round")
    shaded = f"; shaded: {max(hist['train_size'])} px" if big else ""
    style_axis(ax, f"Training loss ({METHOD}, {hist['train_size'][0]} px{shaded})", "epoch",
               f"cross-entropy (label smoothing {cfg.LABEL_SMOOTHING})")
    if accs:
        ax = axes[0][1]
        ep, acc = zip(*accs)
        ax.plot(ep, acc, color=LINE, linewidth=1.5, marker="o", markersize=6, markeredgecolor=SURFACE, markeredgewidth=1.5)
        ax.annotate(f"{acc[-1]:.1f}%", (ep[-1], acc[-1]), textcoords="offset points", xytext=(0, 8), ha="center",
                    color=INK, fontsize=9)
        ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        style_axis(ax, f"Holdout top-1 accuracy at {cfg.TEST_SIZE} px", "epoch", "accuracy (%)")
    fig.tight_layout()
    fig.savefig(os.path.join(run_dir, "curves.png"), dpi=150, facecolor=SURFACE)
    plt.close(fig)


@torch.inference_mode()
def measure_latency(net, batch_size, iters, size, warmup=10):
    """Average seconds per forward pass on random size × size input: model compute only, no image reading or decoding"""
    net.eval()
    x = torch.randn(batch_size, 3, size, size, device=DEVICE)
    for _ in range(warmup):
        with autocast():
            net(x)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        with autocast():
            net(x)
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters


def latency_report(net, cfg):
    """Latency at TEST_SIZE; with flip TTA every image needs two forward passes, so the times are doubled"""
    k = 2 if cfg.TTA_FLIP else 1
    lat_1 = measure_latency(net, 1, 100, cfg.TEST_SIZE) * k
    lat_b = measure_latency(net, 256, 20, cfg.TEST_SIZE) * k
    setting = f"{GPU_NAME}, bfloat16, {cfg.TEST_SIZE} px, {'with' if cfg.TTA_FLIP else 'no'} flip TTA"
    log(f"Inference latency (batch 1): {lat_1 * 1e3:.2f} ms/img | throughput (batch 256): {256 / lat_b:.0f} img/s | {setting}")
    return dict(latency_ms_bs1=lat_1 * 1e3, throughput_img_s=256 / lat_b, latency_setting=setting)


@torch.inference_mode()
def predict(net, items, ctx, cfg):
    """Class predictions at TEST_SIZE; with TTA_FLIP, the logits of the image and of its mirror image are added"""
    net.eval()
    preds = []
    for x, _ in tqdm(make_loader(items, False, cfg.TEST_SIZE, ctx, cfg), desc="predict", leave=False):
        x = to_input(x, ctx)
        with autocast():
            logits = net(x).float()
            if cfg.TTA_FLIP:
                logits += net(torch.flip(x, dims=[3])).float()
        preds.append(logits.argmax(dim=1).cpu())
    return torch.cat(preds).tolist()


def agreement(split, items, preds):
    """Share of images whose new class equals the class in COMPARE_RUN's {split}_team_XX.txt (no labels needed). Two models
    that are each about 76% accurate are both right on at least 52% of the images, so they agree on at least that many"""
    path = os.path.join(OUT_ROOT, "runs", COMPARE_RUN or "-", f"{split}_team_{TEAM_ID}.txt")
    if not COMPARE_RUN or not os.path.exists(path):
        log(f"[{split}] no earlier prediction file to compare with ({path})")
        return None
    with open(path) as f:
        old = dict(line.split()[:2] for line in f if line.strip())
    rate = sum(old.get(p) == str(c) for (p, _), c in zip(items, preds)) / len(items)
    log(f"[{split}] agreement with {COMPARE_RUN}: {rate:.2%} of {len(items):,} images get the same class")
    if rate < 0.5:
        log(f"WARNING: [{split}] fewer than half of the predictions agree with the final model. The new model is clearly "
            "worse, or something is broken (e.g. the class order). Check the log before submitting.")
    return rate


def make_submission(split, items, net, ctx, cfg, run_name, out_dir):
    """Predict one split and write the CodaBench file {split}_team_XX.txt, packed into a zip.
    Returns (zip path, agreement with COMPARE_RUN)"""
    preds = predict(net, items, ctx, cfg)
    assert len(preds) == len(items) == len({p for p, _ in items}), "every image must appear exactly once"
    assert all(0 <= c < cfg.NUM_CLASSES for c in preds)
    name = f"{split}_team_{TEAM_ID}.txt"
    txt_path = os.path.join(out_dir, name)
    with open(txt_path, "w") as f:
        f.writelines(f"{path} {c}\n" for (path, _), c in zip(items, preds))
    zip_path = os.path.join(out_dir, f"submission_{split}_{run_name}.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(txt_path, arcname=name)
    counts = np.bincount(preds, minlength=cfg.NUM_CLASSES)
    log(f"[{split}] {len(preds):,} predictions at {cfg.TEST_SIZE} px over {np.count_nonzero(counts):,} classes "
        f"(max {counts.max()} per class) -> {zip_path}")
    return zip_path, agreement(split, items, preds)

## 9. Full pipeline

`run_experiment` builds the model, initializes the head (skipped when resuming), trains, saves `final.pt`, measures the latency, and writes the submission files and `summary.json`. A run whose `summary.json` already lists every split in `PREDICT_SPLITS` is skipped. `predict_only` loads the `final.pt` of `LOAD_RUN` and only predicts.

In [ ]:
def run_experiment(overrides, run_name):
    """Train one run, then write its submission files. A finished run is skipped, and an interrupted one resumes from
    last.pt. Everything is saved to cs604/runs/<run_name>/ on Drive"""
    global LOG_FILE
    cfg = resolve_config(overrides)
    run_dir = os.path.join(OUT_ROOT, "runs", run_name)
    summary_path = os.path.join(run_dir, "summary.json")
    if os.path.exists(summary_path):
        with open(summary_path) as f:
            previous = json.load(f)
        if all(s in previous.get("submissions", {}) for s in PREDICT_SPLITS):
            log(f"== {run_name}: already finished, skipping")
            return previous
    os.makedirs(run_dir, exist_ok=True)
    LOG_FILE = os.path.join(run_dir, "train_log.txt")
    try:
        with open(os.path.join(run_dir, "config.json"), "w") as f:
            json.dump({**vars(cfg), "METHOD": METHOD, "RUN_NAME": run_name, "BATCH_NAME": BATCH_NAME, "overrides": overrides},
                      f, indent=2)
        log(f"\n========== {run_name} ==========")
        log(f"{METHOD}: lr {cfg.LR}, wd {cfg.WEIGHT_DECAY}, layer decay {cfg.LAYER_DECAY}, drop path {cfg.DROP_PATH}, "
            f"epochs {cfg.EPOCHS}, warmup {cfg.WARMUP_EPOCHS}, holdout {cfg.HOLDOUT_PER_CLASS}/class, prototype head, "
            f"time budget {cfg.TIME_BUDGET_H:.2f} h (course constraint {96 / VRAM_GB:.2f} h ± 10%)")
        log(f"Image sizes: training {cfg.TRAIN_SIZES} px (one per epoch), predictions at {cfg.TEST_SIZE} px"
            + (" with flip TTA" if cfg.TTA_FLIP else ""))
        log(f"Environment: {GPU_NAME} ({VRAM_GB} GB), bfloat16, {os.cpu_count()} CPUs | {VERSIONS}")
        random.seed(cfg.SEED)
        np.random.seed(cfg.SEED)
        torch.manual_seed(cfg.SEED)
        free_gpu()

        model = create_model(cfg)
        ctx = make_ctx(model, cfg)
        clock = Clock()
        resume_ckpt = os.path.join(run_dir, "last.pt")
        resume_ckpt = resume_ckpt if os.path.exists(resume_ckpt) else None
        if resume_ckpt:
            log("Found last.pt: resuming this run (head initialization skipped)")
        else:
            clock.used += init_head(model, ctx, cfg)
        model = apply_method(model, cfg).to(DEVICE)
        params = param_report(model, ctx)
        log(f"Trainable (excl. head): {params['trainable_excl_head']:,} = {params['trainable_pct']:.3f}% of backbone | "
            f"head {ctx.n_head:,} params (fine-tuned)")

        hist, step, spe, speeds = train(model, ctx, cfg, run_dir, clock, resume_ckpt)
        save_state(model, os.path.join(run_dir, "final.pt"), config=vars(cfg))
        with open(os.path.join(run_dir, "history.json"), "w") as f:
            json.dump(hist, f)
        log(f"Training done: {step} steps ({step / spe:.2f} epochs at {hist['train_size']} px), "
            f"{clock.elapsed / 3600:.2f} GPU-hours (course constraint {96 / VRAM_GB:.2f} h ± 10%)")
        # The project doc asks for VRAM in the training log; reserved is closer to what nvidia-smi shows
        peak_gb, reserved_gb = torch.cuda.max_memory_allocated() / 1e9, torch.cuda.max_memory_reserved() / 1e9
        log(f"Peak VRAM: {peak_gb:.1f} GB allocated, {reserved_gb:.1f} GB reserved by PyTorch (GPU has {VRAM_GB} GB)")
        plot_run_curves(hist, cfg, run_dir)

        results = {}
        if ctx.holdout_items:
            results["holdout_top1"], results["holdout_top5"] = evaluate(
                model, make_loader(ctx.holdout_items, False, cfg.TEST_SIZE, ctx, cfg), ctx)
            log(f"Final holdout at {cfg.TEST_SIZE} px (no TTA): top-1 {results['holdout_top1']:.2%}, "
                f"top-5 {results['holdout_top5']:.2%}")
        results.update(latency_report(model, cfg))
        made = {split: make_submission(split, items, model, ctx, cfg, run_name, run_dir) for split, items in PRED_ITEMS.items()}

        summary = dict(run=run_name, method=METHOD, model=cfg.MODEL_NAME, overrides=overrides, train_sizes=hist["train_size"],
                       test_size=cfg.TEST_SIZE, tta_flip=cfg.TTA_FLIP, **params, **results, gpu=GPU_NAME, vram_gb=VRAM_GB,
                       batch=f"{cfg.BATCH_SIZE}x{cfg.ACCUM_STEPS}", epochs_trained=round(step / spe, 2), steps=step,
                       train_speed_img_s=speeds, gpu_hours=clock.elapsed / 3600, budget_hours=cfg.TIME_BUDGET_H,
                       course_constraint_hours=96 / VRAM_GB,
                       peak_vram_gb=peak_gb, peak_vram_reserved_gb=reserved_gb, versions=VERSIONS,
                       submissions={s: z for s, (z, _) in made.items()}, agreement={s: a for s, (_, a) in made.items()})
        with open(summary_path, "w") as f:
            json.dump(summary, f, indent=2)
        log(f"== {run_name} finished: {summary['gpu_hours']:.2f} GPU-h | files to upload: "
            + ", ".join(os.path.basename(z) for z in summary["submissions"].values()))
        return summary
    finally:
        LOG_FILE = None


def predict_only(run_name):
    """Load a trained full fine-tuning run and only predict, at TEST_SIZE (with flip TTA if TTA_FLIP). The files go to a
    new folder runs/<BATCH_NAME>_predict_<run>_<size>px[_tta]/, so the trained run's own folder is never changed"""
    global LOG_FILE
    run_dir = os.path.join(OUT_ROOT, "runs", run_name)
    with open(os.path.join(run_dir, "config.json")) as f:
        saved = json.load(f)
    assert saved.get("METHOD", "full") == "full", f"{run_name} is not a full fine-tuning run"
    cfg = resolve_config({k: saved[k] for k in ("MODEL_NAME", "NUM_CLASSES") if k in saved})
    out_name = f"{BATCH_NAME}_predict_{run_name}_{cfg.TEST_SIZE}px" + ("_tta" if cfg.TTA_FLIP else "")
    out_dir = os.path.join(OUT_ROOT, "runs", out_name)
    os.makedirs(out_dir, exist_ok=True)
    LOG_FILE = os.path.join(out_dir, "predict_log.txt")
    try:
        log(f"Environment: {GPU_NAME} ({VRAM_GB} GB), bfloat16, {os.cpu_count()} CPUs | {VERSIONS}")
        free_gpu()
        model = create_model(cfg)
        ctx = make_ctx(model, cfg)
        ck = torch.load(os.path.join(run_dir, "final.pt"), map_location="cpu", weights_only=False)
        missing, unexpected = model.load_state_dict(ck["state"], strict=False)
        assert not missing and not unexpected, f"missing keys {missing[:5]}, unexpected keys {unexpected[:5]}"
        log(f"Loaded the trained weights from {run_dir}/final.pt; predicting at {cfg.TEST_SIZE} px"
            + (" with flip TTA" if cfg.TTA_FLIP else ""))
        summary = {"method": METHOD}  # keep the trained run's info (training sizes, GPU-hours, ...)
        if os.path.exists(os.path.join(run_dir, "summary.json")):
            with open(os.path.join(run_dir, "summary.json")) as f:
                summary = json.load(f)
        summary.update(run=out_name, source_run=run_name, test_size=cfg.TEST_SIZE, tta_flip=cfg.TTA_FLIP,
                       **latency_report(model, cfg))
        made = {split: make_submission(split, items, model, ctx, cfg, out_name, out_dir) for split, items in PRED_ITEMS.items()}
        summary["submissions"] = {s: z for s, (z, _) in made.items()}
        summary["agreement"] = {s: a for s, (_, a) in made.items()}
        with open(os.path.join(out_dir, "summary.json"), "w") as f:
            json.dump(summary, f, indent=2)
        return summary
    finally:
        LOG_FILE = None

## 10. Run

The run name is `<BATCH_NAME>_1_full_<sizes>`, for example `1105_1400_1_full_224to288`. With `LOAD_RUN` set, nothing is trained and only the predictions are made.

In [ ]:
def size_tag(sizes):
    """Short name of a size schedule for the run folder: [224, ..., 224, 288] → '224to288', [288, 288] → '288'"""
    return "to".join(str(s) for s in dict.fromkeys(sizes))


RESULTS = []
if LOAD_RUN:
    RESULTS.append(predict_only(LOAD_RUN))
else:
    for i, exp in enumerate(EXPERIMENTS, 1):
        RESULTS.append(run_experiment(dict(exp), f"{BATCH_NAME}_{i}_{METHOD}_{size_tag(exp.get('TRAIN_SIZES', TRAIN_SIZES))}"))

## 11. Summary

Saved on Drive in `cs604/batches/<BATCH_NAME>/` as `results.md` (the full data is in `results.json`). "Agreement" is the share of images that get the same class as our final model (`COMPARE_RUN`).

In [ ]:
from IPython.display import Markdown, display


def fmt(v, spec):
    return format(v, spec) if isinstance(v, (int, float)) else "–"


header = ("| Run | Training sizes (px) | Test size | Flip TTA | Holdout top-1 | Epochs | GPU-hours | Peak VRAM (GB) | "
          "Latency bs1 (ms) | Agreement val / test | Test file |\n|---|---|---|---|---|---|---|---|---|---|---|")
rows = [f"| {r['run']} | {', '.join(map(str, r.get('train_sizes', [])))} | {r['test_size']} px | "
        f"{'yes' if r['tta_flip'] else 'no'} | {fmt(r.get('holdout_top1'), '.2%')} | {r.get('epochs_trained', '–')} | "
        f"{fmt(r.get('gpu_hours'), '.2f')} | {fmt(r.get('peak_vram_gb'), '.1f')} | {fmt(r.get('latency_ms_bs1'), '.2f')} | "
        f"{fmt(r['agreement'].get('val'), '.2%')} / {fmt(r['agreement'].get('test'), '.2%')} | "
        f"{os.path.basename(r['submissions'].get('test', '–'))} |" for r in RESULTS]
table = header + "\n" + "\n".join(rows)
with open(os.path.join(BATCH_DIR, "results.md"), "w") as f:
    f.write(table + "\n")
with open(os.path.join(BATCH_DIR, "results.json"), "w") as f:
    json.dump(RESULTS, f, indent=2)
log("\n" + table)
display(Markdown(table))

## 12. Auto disconnect (optional)

All results are already saved on Drive. `DISCONNECT_WHEN_DONE = True` releases the runtime after checking that everything is written, so it stops using compute units. Leave it off if you still want to run other experiments in the same session.

In [ ]:
if DISCONNECT_WHEN_DONE:
    from google.colab import runtime
    log("All done; releasing the runtime ...")
    drive.flush_and_unmount()  # make sure all results are written to Drive
    runtime.unassign()

## Next steps: the unranked test submission

1. In the log, check the `[test]` lines: 100,000 predictions, and the agreement with our final model (a working model agrees on well over 52% of the images).
2. Download `submission_test_<run>.zip` from `cs604/runs/<run>/`, and upload it once to Phase 2 on CodaBench (due 06 Nov 2026, 23:59).
3. Right after the upload, check the leaderboard. It must still show our ranked result, 76.09% from `1019_1319_1_full`. If the new submission replaced it, ask the TA to restore it, as agreed.
4. Keep the TA's confirmation (for example a screenshot) with the project files.
5. In the report and the slides, list the new score as a post-deadline, unranked experiment approved by the TA, next to the ranked 76.09%. State the training sizes, the test size, the GPU-hours (with the default plan about 1.7, over the 1.2-hour constraint) and the latency with TTA (all in `results.md`).